<a href="https://colab.research.google.com/github/chakma21/YTube_RAG/blob/colab/YouTube_QA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  YouTube Video Q&A — RAG System

### Pipeline overview
```
YouTube URL
    │
    ▼
Transcript  ──(youtube-transcript-api)──▶  segments [{text, start}]
    │  (fallback) yt-dlp + Whisper-small
    ▼
Chunker     ──(~300 words, 50-word overlap)──▶  chunks
    │
    ▼
Embedder    ──(all-MiniLM-L6-v2 · 384-dim)──▶  float32 matrix
    │
    ▼
FAISS Index ──(IndexFlatIP · cosine sim)──▶  vector store
    │
    ▼  [query time]
Retriever   ──(top-k chunks)──────────────▶  context
    │
    ▼
Generator   ──(google/flan-t5-base)────────▶  answer
```

## 1 · Install dependencies

In [ ]:
!pip install -q \
    youtube-transcript-api \
    yt-dlp \
    transformers \
    sentence-transformers \
    faiss-cpu \
    torch \
    sentencepiece \
    accelerate

print('✅ All packages installed')

✅ All packages installed


## 2 · Paste the pipeline code

In [ ]:
# ─── If running in Colab, either:
#   a) Upload rag_pipeline.py and run:  from rag_pipeline import YouTubeRAG
#   b) Or paste the full rag_pipeline.py source below this cell
#
# For a quick start, we import directly:

import sys, os

# If you cloned/uploaded the project files:
# sys.path.insert(0, '/content/yt_rag')

# ── Inline minimal import (works without uploading files) ──────────────────
# Copy-paste rag_pipeline.py content here OR run the cell below to download it

print('Ready to import YouTubeRAG')

Ready to import YouTubeRAG


In [ ]:
# ── Quick inline pipeline (self-contained, no file upload needed) ──────────

import re, os, numpy as np
from typing import Optional
chat_history=[]

# ── Transcript ──
def extract_video_id(url):
    m = re.search(r'(?:v=|youtu\.be/|embed/|shorts/)([A-Za-z0-9_-]{11})', url)
    return m.group(1) if m else None

def fetch_transcript(video_url):
    vid_id = extract_video_id(video_url)
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        raw = YouTubeTranscriptApi.get_transcript(vid_id)
        segments = [{'text': s['text'], 'start': s['start']} for s in raw]
        print(f' Transcript via YouTube API ({len(segments)} segments)')
        return segments, True
    except Exception as e:
        print(f'  Falling back to Whisper ({e})')
        return transcribe_with_whisper(video_url), False

def transcribe_with_whisper(video_url):
    import yt_dlp
    from transformers import pipeline as hf_pipeline
    audio_path = '/tmp/yt_audio.mp3'
    ydl_opts = {'format':'bestaudio/best','outtmpl':'/tmp/yt_audio',
                'postprocessors':[{'key':'FFmpegExtractAudio','preferredcodec':'mp3','preferredquality':'128'}],'quiet':True}
    print('⬇️  Downloading audio…')
    with yt_dlp.YoutubeDL(ydl_opts) as ydl: ydl.download([video_url])
    print('🎙️  Transcribing with Whisper-small…')
    asr = hf_pipeline('automatic-speech-recognition', model='openai/whisper-small', return_timestamps=True)
    result = asr(audio_path, generate_kwargs={"task": "translate"})
    if isinstance(result.get('chunks'), list):
        segs = [{'text':c['text'],'start':c['timestamp'][0] or 0.} for c in result['chunks']]
    else:
        segs = [{'text':result['text'],'start':0.}]
    if os.path.exists(audio_path): os.remove(audio_path)
    print(f'✅ Whisper done ({len(segs)} segments)')
    return segs

# ── Chunking ──

import re

def clean_text(text):
    text = re.sub(r'(.)\1{4,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def split_into_chunks(segments, words_per_chunk=300):
    chunks, cur_words, cur_start, idx = [], [], 0., 0
    for seg in segments:
        text = clean_text(seg['text'])
        words = text.split()
        text = clean_text(seg['text'])
        if not text:
            continue
        if not cur_words: cur_start = seg.get('start', 0.)
        cur_words.extend(words)
        if len(cur_words) >= words_per_chunk:
            chunks.append({'text':' '.join(cur_words),'start':cur_start,'chunk_index':idx})
            idx += 1
            cur_words = cur_words[-50:]
            cur_start = seg.get('start', 0.)
    if cur_words: chunks.append({'text':' '.join(cur_words),'start':cur_start,'chunk_index':idx})
    print(f'✅ {len(chunks)} chunks created')
    return chunks

# ── Embeddings ──
def load_embedding_model():
    from sentence_transformers import SentenceTransformer
    print(' Loading all-MiniLM-L6-v2…')
    m = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    print('✅ Embedding model ready')
    return m

def embed_chunks(chunks, embed_model):
    texts = [c['text'] for c in chunks]
    print(f'🔢 Embedding {len(texts)} chunks…')
    embs = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True).astype('float32')
    print(f'✅ Embeddings: {embs.shape}')
    return embs

# ── FAISS ──
def build_faiss_index(embeddings):
    import faiss
    faiss.normalize_L2(embeddings)
    idx = faiss.IndexFlatIP(embeddings.shape[1])
    idx.add(embeddings)
    print(f'✅ FAISS index: {idx.ntotal} vectors')
    return idx

# ── Retrieval ──
def retrieve_relevant_chunks(query, embed_model, index, chunks, top_k=4):
    import faiss
    qv = embed_model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(qv)
    scores, idxs = index.search(qv, top_k)
    results = []
    for s, i in zip(scores[0], idxs[0]):
        if i == -1: continue
        c = dict(chunks[i]); c['score'] = float(s); results.append(c)
    return results

# ── Generation ──
def load_generation_model():
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    model_name = "mistralai/Mistral-7B-Instruct-v0.1"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print('✅ Generation model ready')
    return tokenizer, model

def format_timestamp(s):
    s = int(s); h, r = divmod(s, 3600); m, sec = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{sec:02d}' if h else f'{m:02d}:{sec:02d}'

def generate_answer(query, chunks, tokenizer, model):
    import torch

    # Clean context
    ctx = "\n\n".join([c["text"][:300] for c in chunks])

    prompt = f"""
<s>[INST]
You are a helpful and intelligent assistant.

Understand the context and answer clearly.

Rules:
- Do NOT copy text
- Explain meaning in simple words
- Answer ONLY the question asked
- Do NOT repeat or re-explain previous concepts
- Do NOT add unnecessary background
- Ensure the answer ends properly and is not cut off
- If it's a concept → explain
- If it's a story → summarize
- If it's music → say it's a music video
- Answer clearly in 2–3 sentences

Context:
{ctx}

Question:
{query}
[/INST]
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.split("[/INST]")[-1].strip()

    return {"short_answer": answer}


def format_history(chat_history):
    formatted = []
    for i, msg in enumerate(chat_history):
        role = "User" if i % 2 == 0 else "Assistant"
        formatted.append(f"{role}: {msg}")
    return "\n".join(formatted)


print('✅ All pipeline functions defined')


✅ All pipeline functions defined


## 3 · Load models (once per session)

In [ ]:
embed_model = load_embedding_model()
tokenizer, gen_model = load_generation_model()
print('\n🚀 All models loaded and ready!')

 Loading all-MiniLM-L6-v2…


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Generation model ready

🚀 All models loaded and ready!


## 4 · Ingest a YouTube video

In [ ]:
# 🔗 Change this to any YouTube URL
VIDEO_URL = 'https://youtu.be/FU09dmtIoIc?si=0c9wlD7lJyi8ZS_b'  # replace me

# Step 1: Transcript
segments, has_timestamps = fetch_transcript(VIDEO_URL)
print(f'\nFirst segment: {segments[0]}')

# Step 2: Chunk
chunks = split_into_chunks(segments, words_per_chunk=300)

# Step 3: Embed
embeddings = embed_chunks(chunks, embed_model)

# Step 4: FAISS index
faiss_index = build_faiss_index(embeddings)

print(f'\n✅ Pipeline complete — {len(chunks)} chunks indexed')

  Falling back to Whisper (type object 'YouTubeTranscriptApi' has no attribute 'get_transcript')
⬇️  Downloading audio…


🎙️  Transcribing with Whisper-small…


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Whisper done (91 segments)

First segment: {'text': ' So hi guys, welcome back to my youtube channel, I hope you all are doing well, and as you can see, we are sitting in a beautiful view and I was reading a book, from which a term has been found, the wild horse effect, this is a term used in psychology, 90% people are affected by this particular problem and its solution will come to you here, so first of all, today we will talk about the triggers in mental health, you all must be some kind of triggers, and whenever that If someone tries to scratch a particular safe space and tries to get you out of that safe space then you suddenly get angry.', 'start': 0.0}
✅ 6 chunks created
🔢 Embedding 6 chunks…


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings: (6, 384)
✅ FAISS index: 6 vectors

✅ Pipeline complete — 6 chunks indexed


## 5 · Ask questions

In [ ]:
# ❓ Change the question below
QUESTION = 'What is the main topic of this video?'
TOP_K    = 4        # number of chunks to retrieve
DETAILED = False    # set True for a longer answer


# Retrieve + Generate
relevant = retrieve_relevant_chunks(QUESTION, embed_model, faiss_index, chunks, top_k=TOP_K)
# result   = generate_answer(QUESTION, relevant, tokenizer, gen_model, detailed=DETAILED)
# Combine history + current question
MAX_HISTORY = 4  # last 2 Q&A
history_text = format_history(chat_history[-4:])

# final_query = f"""
# Conversation so far:
# {history_text}

# Current question:
# {QUESTION}
# """
if len(chat_history) > 0:
    history_text = format_history(chat_history[-4:])
    final_query = f"""
Conversation so far:
{history_text}

Current question:
{QUESTION}
"""
else:
    final_query = QUESTION

result = generate_answer(
    final_query,
    relevant,
    tokenizer,
    gen_model,
)

chat_history.append(QUESTION)
chat_history.append(result["short_answer"])

print("\n[DEBUG] Relevant chunks:\n")
for c in relevant:
    print(c['text'][:200])

# ── Display ──────────────────────────────────────────────────────────────────
print('=' * 20)
print(f'Question : {QUESTION}')
print('=' * 20)
# print(f'Answer   : {result["short_answer"]}')
print("RAW RESULT:", result)
print(f'Answer   : {result.get("short_answer", "No answer generated")}')




[DEBUG] Relevant chunks:

species called a vampire bat. She comes and while resting the horses, she basically stings them and kills them. But she doesn't do much damage. But even then, the horse feels that it has been a huge l
and let go of him, then I will be able to heal that wound quickly. And I will be able to move forward in life with my experiences. So this was the video for today. And I would just like to end the vid
So hi guys, welcome back to my youtube channel, I hope you all are doing well, and as you can see, we are sitting in a beautiful view and I was reading a book, from which a term has been found, the wi
around a tree or an object, or it rolls itself on the ground to digest it. And if someone has to strangle it, it rolls itself. So it rolls itself around the R e to destroy 100. That is to destroy the 
Question : What is the main topic of this video?
RAW RESULT: {'short_answer': 'The main topic of this video is the concept of the "wild horse effect" in psychology, whic

## 6 · (Optional) Interactive loop
Run this cell to keep asking questions about the same video.

In [ ]:
print('Type your question and press Enter. Type "quit" to stop.\n')
while True:
    q = input('❓ Your question: ').strip()
    if q.lower() in ('quit', 'exit', 'q', ''): break
    rel = retrieve_relevant_chunks(q, embed_model, faiss_index, chunks, top_k=4)
    ans = generate_answer(q, rel, tokenizer, gen_model)
    print(f'\n✅ Answer: {ans["short_answer"]}\n')
    print('-' * 50)

Type your question and press Enter. Type "quit" to stop.

❓ Your question: explain the topic in detailed


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



✅ Answer: The Wild Horse Effect is a term used in psychology that refers to a phenomenon where a person experiences a strong emotional reaction to a situation, even if the situation itself is not particularly significant or dangerous. This effect is often seen in situations where a person feels threatened or vulnerable, and can lead to feelings of fear, anxiety, and even panic.

The term "wild horse effect" is often used to describe the emotional response that a person may have when they feel that their sense of safety or security has been threatened. For example, if a person is walking through a dark alley at night and suddenly hears a loud noise, they may experience a wild horse effect, feeling a sudden surge of fear and anxiety even though there may not be

--------------------------------------------------
❓ Your question: how to reduce it


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



✅ Answer: To reduce excessive emotional reaction due to a wound, it is important to acknowledge and process the emotions, rather than suppressing or denying them. It may also be helpful to seek support from others, such as friends, family, or a therapist, to help work through the emotions and gain perspective. Additionally, practicing self-care, such as getting enough sleep, exercise, and healthy nutrition, can also help reduce the emotional impact of a wound.

--------------------------------------------------
❓ Your question: quit
